# 10 — No-Show / Abandonment Risk

A GradientBoosting classifier predicting whether a ticket ends in no-show / walk-away, from what
is known while the customer waits: time, service, queue length, channel, and holiday pressure.
Validated on a time-based holdout (ROC-AUC).

**Outputs the `no_show_risk` insight** consumed by the Staff & Manager dashboards.

## 1. Setup & data

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams.update({"axes.titleweight": "bold", "axes.titlesize": 12, "figure.dpi": 110})
pd.set_option("display.max_columns", 40)

# QMe Now palette (matches the admin dashboards)
NAVY, STEEL, TEAL, RED, GOLD = "#2F5063", "#6E8AA6", "#2E7387", "#B23A4E", "#9A6B2E"
BLUES = sns.light_palette(NAVY, n_colors=6, reverse=True)

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE))
DOW = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
def hour_label(h): return f"{((int(h) + 11) % 12) + 1}{'am' if h < 12 else 'pm'}"
print("Ready.")

In [ ]:
from scripts import predict_no_show as ns

conn = ns.connect()
df = ns.load_records(conn)
rate = df["abandoned"].mean() * 100
print(f"{len(df):,} tickets · overall abandonment {rate:.1f}%")

bal = df["abandoned"].map({0: "Served", 1: "No-show / left"}).value_counts()
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(bal.index, bal.values, color=[TEAL, RED])
ax.set_title("Outcome balance"); ax.set_ylabel("Tickets")
for i, v in enumerate(bal.values): ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Train + validate\nTime-based holdout — the recent 20% of visits are held out for scoring.

In [ ]:
trained = ns.train(df)
print(f"ROC-AUC on holdout: {trained['auc']}  ({trained['test_rows']:,} rows, {trained['test_window']})")
imp = pd.DataFrame(trained["importances"])
DRIVER = {"queue_length": "Line length at join", "hour": "Time of day", "service_enc": "Which service",
          "dow": "Day of week", "is_walk_in": "Walk-in vs app", "is_holiday": "Holiday",
          "is_month_end": "Month-end", "month": "Month", "branch_enc": "Which branch"}
imp["label"] = imp["feature"].map(lambda f: DRIVER.get(f, f))
imp

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
d = imp.sort_values("importance")
ax.barh(d["label"], d["importance"], color=NAVY)
ax.set_title("What Drives No-Show Risk (feature importance)"); ax.set_xlabel("Importance")
plt.tight_layout(); plt.show()

## 3. Risk lookup — service × channel × queue length\nThe compact table the staff dashboard reads to flag at-risk tickets.

In [ ]:
df_biz = df[df["business_id"] == df["business_id"].mode().iloc[0]]
services = ns.risk_table(trained, df_biz)
svc = services[0]
grid = pd.DataFrame(svc["cells"])
heat = grid.pivot(index="channel", columns="queue_max", values="risk_pct")
print(f"Service: {svc['service_name']} · observed abandonment {svc['observed_abandon_rate_pct']}%")

fig, ax = plt.subplots(figsize=(9, 2.6))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "Risk %"}, ax=ax)
ax.set_title(f"Predicted No-Show Risk — {svc['service_name']}")
ax.set_xlabel("Queue length at join (≤)"); ax.set_ylabel("Channel")
plt.tight_layout(); plt.show()
heat.round(1)

In [ ]:
if os.getenv("WRITE_DB") == "1":
    ins, gen, stale = ns.build_insights(df, trained)
    ns.upsert_insights(conn, ins, gen, stale, ns.MODEL_VERSION)
    print(f"Upserted {len(ins)} no_show_risk insight(s).")
else:
    print("Preview only — set WRITE_DB=1 to persist.")
conn.close()